In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# =========================
# CONFIG
# =========================
BASE_TFM = Path("/content/drive/MyDrive/TFM")

DATASET_DIR = BASE_TFM / "datasets" / "videos" / "Gun_Action_Recognition_Dataset"
META_DIR    = DATASET_DIR / "metadata"
SPLITS_DIR  = DATASET_DIR / "splits"

INDEX_CSV   = META_DIR / "index_handgun.csv"
META_ACTIONS_CSV = META_DIR / "metadata_actions.csv"

SEED = 42
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
TEST_FRAC  = 0.15

# Si quieres “mini benchmark” ya fijo, puedes poner aquí un número máximo para test/val:
MAX_TEST = None   # ej: 30
MAX_VAL  = None   # ej: 30

# =========================
# LOAD
# =========================
assert INDEX_CSV.exists(), f"No existe {INDEX_CSV}. Ejecuta antes la celda de index."
df = pd.read_csv(INDEX_CSV)

# Solo clips con video y label si quieres (recomendado para evaluación cuantitativa)
df = df[df["video_path"].notna() & (df["video_path"].astype(str).str.len() > 0)].copy()

# Extraer code (prefijo antes del primer "_": PAH1, PCH3, etc.)
df["code"] = df["clip_id"].astype(str).str.split("_").str[0]

# Añadir "action" si el code está en metadata_actions
action_map = {}
if META_ACTIONS_CSV.exists():
    df_meta = pd.read_csv(META_ACTIONS_CSV)
    action_map = dict(zip(df_meta["code"], df_meta["action"]))

df["action"] = df["code"].map(action_map)
df["group"] = df["action"].fillna(df["code"])  # si no hay action, usa code como grupo

print("Clips totales:", len(df))
print("Distribución por action (si existe):")
print(df["action"].value_counts(dropna=False))

# =========================
# STRATIFIED SPLIT
# =========================
rng = np.random.default_rng(SEED)

train_idx = []
val_idx   = []
test_idx  = []

for g, gdf in df.groupby("group"):
    idx = gdf.index.to_numpy()
    rng.shuffle(idx)

    n = len(idx)
    n_train = int(round(n * TRAIN_FRAC))
    n_val   = int(round(n * VAL_FRAC))
    # Asegurar que suma bien y que test >= 1 si hay suficientes
    n_test  = n - n_train - n_val
    if n >= 3:
        n_test = max(n_test, 1)
        # reajusta si nos pasamos
        while n_train + n_val + n_test > n:
            if n_train > 1:
                n_train -= 1
            elif n_val > 1:
                n_val -= 1
            else:
                break
    else:
        # grupos muy pequeños: mete todo a train
        n_train, n_val, n_test = n, 0, 0

    train_idx.extend(idx[:n_train].tolist())
    val_idx.extend(idx[n_train:n_train+n_val].tolist())
    test_idx.extend(idx[n_train+n_val:n_train+n_val+n_test].tolist())

df_train = df.loc[train_idx].sample(frac=1, random_state=SEED)  # mezclar global
df_val   = df.loc[val_idx].sample(frac=1, random_state=SEED)
df_test  = df.loc[test_idx].sample(frac=1, random_state=SEED)

# Aplicar límites opcionales para mini benchmark
if MAX_VAL is not None and len(df_val) > MAX_VAL:
    df_val = df_val.sample(n=MAX_VAL, random_state=SEED)

if MAX_TEST is not None and len(df_test) > MAX_TEST:
    df_test = df_test.sample(n=MAX_TEST, random_state=SEED)

# Sanity: no solapes
assert set(df_train.index).isdisjoint(set(df_val.index))
assert set(df_train.index).isdisjoint(set(df_test.index))
assert set(df_val.index).isdisjoint(set(df_test.index))

print("\nSplit sizes:")
print(" - train:", len(df_train))
print(" - val  :", len(df_val))
print(" - test :", len(df_test))

print("\nDistribución por action en cada split:")
for name, dfx in [("train", df_train), ("val", df_val), ("test", df_test)]:
    print(f"\n{name}:")
    print(dfx["action"].value_counts(dropna=False))

# =========================
# SAVE .txt (rutas a video.mp4)
# =========================
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

train_txt = SPLITS_DIR / "handgun_train.txt"
val_txt   = SPLITS_DIR / "handgun_val.txt"
test_txt  = SPLITS_DIR / "handgun_test.txt"

train_txt.write_text("\n".join(df_train["video_path"].astype(str).tolist()) + "\n", encoding="utf-8")
val_txt.write_text("\n".join(df_val["video_path"].astype(str).tolist()) + "\n", encoding="utf-8")
test_txt.write_text("\n".join(df_test["video_path"].astype(str).tolist()) + "\n", encoding="utf-8")

print("\n✅ Splits guardados en:")
print(" -", train_txt)
print(" -", val_txt)
print(" -", test_txt)

# (Opcional) guardo también un resumen CSV por si quieres inspeccionarlo rápido
summary_csv = SPLITS_DIR / "handgun_splits_summary.csv"
out = pd.concat([
    df_train.assign(split="train"),
    df_val.assign(split="val"),
    df_test.assign(split="test")
], axis=0)
out.to_csv(summary_csv, index=False)
print("✅ Resumen:", summary_csv)

Clips totales: 141
Distribución por action (si existe):
action
Aim       80
Carry     55
Reload     6
Name: count, dtype: int64

Split sizes:
 - train: 98
 - val  : 21
 - test : 22

Distribución por action en cada split:

train:
action
Aim       56
Carry     38
Reload     4
Name: count, dtype: int64

val:
action
Aim       12
Carry      8
Reload     1
Name: count, dtype: int64

test:
action
Aim       12
Carry      9
Reload     1
Name: count, dtype: int64

✅ Splits guardados en:
 - /content/drive/MyDrive/TFM/datasets/videos/Gun_Action_Recognition_Dataset/splits/handgun_train.txt
 - /content/drive/MyDrive/TFM/datasets/videos/Gun_Action_Recognition_Dataset/splits/handgun_val.txt
 - /content/drive/MyDrive/TFM/datasets/videos/Gun_Action_Recognition_Dataset/splits/handgun_test.txt
✅ Resumen: /content/drive/MyDrive/TFM/datasets/videos/Gun_Action_Recognition_Dataset/splits/handgun_splits_summary.csv
